# Stage 3 — Local anomaly and standardised-enhancement workflow

This notebook implements Stage 3 of the thesis workflow.

It reproduces the **local anomaly and enhancement** branch:

1. access monthly Sentinel-5P/TROPOMI XCH4 data from Google Earth Engine;
2. export monthly mean XCH4 GeoTIFFs over CONUS with external support around the boundary;
3. compute local-background anomaly and standardised enhancement for multiple kernel widths;
4. create binary enhancement masks using \(Z \ge 2\);
5. sample Carbon Mapper plume-event coordinates against the enhancement masks;
6. generate summary tables and publication-style diagnostic figures.

The workflow keeps the measurement meanings separate:

- Sentinel-5P/TROPOMI: monthly XCH4 raster fields in ppb;
- local anomaly: \(\Delta XCH4 = XCH4 - B_k\), in ppb;
- standardised enhancement: \(Z_k = \Delta XCH4 / \sigma_k\), dimensionless;
- Carbon Mapper: plume-event point records, not continuous source inventories.

The point-to-mask comparison is a **kernel-sensitivity test**, not a Sentinel-5P plume-detection-rate estimate.


In [ ]:
# Optional Colab dependency installation
# Run only when needed.

!pip -q install earthengine-api geemap rasterio geopandas shapely pyproj pandas numpy matplotlib torch tqdm

In [ ]:
# Optional Google Drive mount for Colab

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Imports

import os
import re
import gc
import math
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rasterio
from rasterio.windows import Window
from rasterio.transform import rowcol
from rasterio.plot import show

try:
    import torch
    import torch.nn.functional as F
except Exception as exc:
    torch = None
    F = None
    print("PyTorch is not available:", exc)

try:
    import ee
except Exception as exc:
    ee = None
    print("Earth Engine is not available:", exc)


## 1. User configuration

The principal analysis parameters are:

- `BUFFER_DEG = 2.5`: external support around CONUS for local-background
  estimation;
- `KERNEL_DEG_LIST = [1, 2, 3, 4, 5, 6]`: local-background window widths;
- `ENH_THRESHOLD = 2.0`: standardized-enhancement threshold for the binary
  mask.


In [ ]:
# ============================================================
# USER SETTINGS
# ============================================================

# Earth Engine / Sentinel-5P
EE_PROJECT = None  # e.g. "your-ee-project"; keep None if default auth works
S5P_COLLECTION = "COPERNICUS/S5P/OFFL/L3_CH4"
PREFERRED_BAND = "CH4_column_volume_mixing_ratio_dry_air_bias_corrected"
FALLBACK_BAND = "CH4_column_volume_mixing_ratio_dry_air"

# Time range
YEARS = [2023, 2024, 2025]
MONTHS = list(range(1, 13))

# Study area and raster support
BUFFER_DEG = 2.5
M_PER_DEG = 111320.0
BUFFER_M = BUFFER_DEG * M_PER_DEG

# Local-background kernels, full window width in degrees
KERNEL_DEG_LIST = [1, 2, 3, 4, 5, 6]

# Enhancement threshold
SIGMA_MIN = 0.1
ENH_THRESHOLD = 2.0

# Google Drive folders
DRIVE_EXPORT_FOLDER_MEAN = "stage3_XCH4_mean"
DRIVE_EXPORT_FOLDER_GEE_PRODUCTS = "stage3_gee_anomaly_enhancement"

# Local/Drive paths after export/download
ROOT = Path("/content/drive/MyDrive/stage3_local_anomaly")
XCH4_MEAN_DIR = ROOT / "XCH4_mean"
LOCAL_PRODUCTS_DIR = ROOT / "local_products"
SUMMARY_DIR = ROOT / "summary_tables"
FIGURE_DIR = ROOT / "figures"

for d in [XCH4_MEAN_DIR, LOCAL_PRODUCTS_DIR, SUMMARY_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Expected input monthly mean filename:
# XCH4_mean_YYYY_MM_CONUSbuf2p5deg.tif
MEAN_TEMPLATE = "XCH4_mean_{year}_{month:02d}_CONUSbuf2p5deg.tif"

# Output filenames from local processor
ANOM_TEMPLATE = "anomaly_{year}_{month:02d}_{kernel}deg_CONUSbuf2p5deg.tif"
SIGMA_TEMPLATE = "sigma_{year}_{month:02d}_{kernel}deg_CONUSbuf2p5deg.tif"
Z_TEMPLATE = "enhancementZ_{year}_{month:02d}_{kernel}deg_CONUSbuf2p5deg.tif"
MASK_TEMPLATE = "enhancement_ge2_{year}_{month:02d}_{kernel}deg_CONUSbuf2p5deg.tif"

# Carbon Mapper plume CSV for point-to-mask comparison
# Must contain longitude/latitude and timestamp columns.
PLUMES_CSV = "/content/drive/MyDrive/stage3_local_anomaly/carbonmapper_CH4_plumes_CONUS_2023_2025.csv"

# Example case used in thesis: Visalia area, October 2024.
CASE_YEAR = 2024
CASE_MONTH = 10
CASE_NAME = "Visalia_October_2024"

# Bounding box used only to select/plot the case from the plume CSV.
# Format: [min_lon, min_lat, max_lon, max_lat]
CASE_BBOX = [-120.1, 35.9, -118.8, 36.9]

print("Configured folders:")
print("  ROOT:", ROOT)
print("  XCH4_MEAN_DIR:", XCH4_MEAN_DIR)
print("  LOCAL_PRODUCTS_DIR:", LOCAL_PRODUCTS_DIR)
print("  SUMMARY_DIR:", SUMMARY_DIR)
print("  FIGURE_DIR:", FIGURE_DIR)


## 2. Earth Engine authentication and CONUS geometry

This section initializes Earth Engine and constructs a strict CONUS geometry from `TIGER/2018/States`, excluding Alaska, Hawaii and territories. A support geometry is created by buffering CONUS by 2.5° equivalent distance. The support buffer reduces edge effects in local-background calculations.


In [ ]:
# ============================================================
# EARTH ENGINE INITIALIZATION
# ============================================================

if ee is None:
    raise ImportError("earthengine-api is not installed. Run the dependency cell first.")

try:
    if EE_PROJECT:
        ee.Initialize(project=EE_PROJECT)
    else:
        ee.Initialize()
except Exception:
    ee.Authenticate()
    if EE_PROJECT:
        ee.Initialize(project=EE_PROJECT)
    else:
        ee.Initialize()

# Strict CONUS geometry
states = ee.FeatureCollection("TIGER/2018/States")
exclude = ee.List(["AK", "HI", "PR", "GU", "VI", "AS", "MP"])
conus_fc = states.filter(ee.Filter.Not(ee.Filter.inList("STUSPS", exclude)))
conus_geom = conus_fc.geometry()
conus_support_geom = conus_geom.buffer(BUFFER_M)

print("Earth Engine initialized.")
print("Using strict CONUS + support buffer of", BUFFER_DEG, "degrees.")


## 3. Export monthly mean Sentinel-5P XCH4 GeoTIFFs from GEE

This step exports the monthly mean XCH4 rasters used as the basis for anomaly and enhancement calculation.

The selected Earth Engine collection is:

```text
COPERNICUS/S5P/OFFL/L3_CH4
```

Preferred band:

```text
CH4_column_volume_mixing_ratio_dry_air_bias_corrected
```

The output rasters are exported over the CONUS support geometry. They are later processed locally to compute local background, anomaly, sigma and enhancement masks.


In [ ]:
# ============================================================
# EXPORT MONTHLY MEAN XCH4 FROM EARTH ENGINE
# ============================================================

base_ic = ee.ImageCollection(S5P_COLLECTION)
first_img = ee.Image(base_ic.first())
band_names = first_img.bandNames().getInfo()

if PREFERRED_BAND in band_names:
    CH4_BAND = PREFERRED_BAND
elif FALLBACK_BAND in band_names:
    CH4_BAND = FALLBACK_BAND
else:
    raise ValueError(f"No methane band found. Available bands: {band_names}")

EXPORT_SCALE_M = first_img.select(CH4_BAND).projection().nominalScale().getInfo()

print("Using CH4 band:", CH4_BAND)
print("Nominal export scale:", EXPORT_SCALE_M, "m")

def masked_empty_image(name):
    return (
        ee.Image.constant(0)
        .rename(name)
        .toFloat()
        .updateMask(ee.Image.constant(0))
        .clip(conus_support_geom)
    )

def monthly_mean_xch4(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, "month")

    col = (
        ee.ImageCollection(S5P_COLLECTION)
        .filterDate(start, end)
        .filterBounds(conus_support_geom)
        .select(CH4_BAND)
    )

    mean_img = col.mean().rename("XCH4_mean").toFloat().clip(conus_support_geom)
    out_img = ee.Image(
        ee.Algorithms.If(
            col.size().gt(0),
            mean_img,
            masked_empty_image("XCH4_mean")
        )
    )
    return out_img

def start_monthly_mean_exports(years=YEARS, months=MONTHS):
    tasks = []
    for year in years:
        for month in months:
            out_name = f"XCH4_mean_{year}_{month:02d}_CONUSbuf2p5deg"
            img = monthly_mean_xch4(year, month)

            task = ee.batch.Export.image.toDrive(
                image=img,
                description=out_name,
                folder=DRIVE_EXPORT_FOLDER_MEAN,
                fileNamePrefix=out_name,
                region=conus_support_geom,
                scale=EXPORT_SCALE_M,
                maxPixels=1e13,
                fileFormat="GeoTIFF",
            )
            task.start()
            tasks.append(task)
            print("Started export:", out_name)

    print(f"Submitted {len(tasks)} monthly mean export tasks to Drive folder:", DRIVE_EXPORT_FOLDER_MEAN)
    return tasks

# Uncomment to start all exports.
# mean_export_tasks = start_monthly_mean_exports()
# [t.status() for t in mean_export_tasks[:3]]


## 4. Optional: GEE direct export of anomaly/sigma/enhancement products

This optional cell computes local median, anomaly, sigma, and enhancement directly in Earth Engine. It is suitable for smaller kernel sets or one-month tests. The local tiled processor in Section 5 supports larger kernels and full monthly archives.


In [ ]:
# ============================================================
# OPTIONAL GEE DIRECT EXPORT FOR ONE MONTH AND MULTIPLE KERNELS
# ============================================================

def make_ee_kernel(kernel_deg):
    radius_m = (kernel_deg / 2.0) * M_PER_DEG
    return ee.Kernel.square(radius=radius_m, units="meters", normalize=False)

def gee_monthly_local_products(year, month, kernel_deg):
    kernel = make_ee_kernel(kernel_deg)
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, "month")

    col = (
        ee.ImageCollection(S5P_COLLECTION)
        .filterDate(start, end)
        .filterBounds(conus_support_geom)
        .select(CH4_BAND)
    )

    monthly = col.mean().rename("XCH4_mean").toFloat().clip(conus_support_geom)

    local_median = monthly.reduceNeighborhood(
        reducer=ee.Reducer.median(),
        kernel=kernel,
        skipMasked=True
    ).rename("local_median").toFloat()

    anomaly = monthly.subtract(local_median).rename("anomaly").toFloat()

    # Local sigma approximation:
    # cap values above the local p95 before calculating local standard deviation.
    local_p95 = monthly.reduceNeighborhood(
        reducer=ee.Reducer.percentile([95]),
        kernel=kernel,
        skipMasked=True
    ).rename("local_p95").toFloat()

    xch4_capped = monthly.min(local_p95).rename("XCH4_capped").toFloat()

    sigma = xch4_capped.reduceNeighborhood(
        reducer=ee.Reducer.stdDev(),
        kernel=kernel,
        skipMasked=True
    ).rename("sigma").toFloat()

    z = anomaly.divide(sigma.max(SIGMA_MIN)).rename("enhancementZ").toFloat()
    mask = z.gte(ENH_THRESHOLD).selfMask().rename("enhancement_ge2").toByte()

    has_data = col.size().gt(0)
    anomaly = ee.Image(ee.Algorithms.If(has_data, anomaly, masked_empty_image("anomaly")))
    sigma = ee.Image(ee.Algorithms.If(has_data, sigma, masked_empty_image("sigma")))
    z = ee.Image(ee.Algorithms.If(has_data, z, masked_empty_image("enhancementZ")))
    mask = ee.Image(ee.Algorithms.If(has_data, mask, masked_empty_image("enhancement_ge2").toByte()))

    return anomaly.clip(conus_support_geom), sigma.clip(conus_support_geom), z.clip(conus_support_geom), mask.clip(conus_support_geom)

def start_gee_product_exports_one_month(year, month, kernel_list=(1,2,3,4,5,6)):
    tasks = []
    for kernel_deg in kernel_list:
        kernel_label = f"{int(kernel_deg)}deg"
        anomaly, sigma, z, mask = gee_monthly_local_products(year, month, kernel_deg)

        exports = [
            (anomaly, f"anomaly_{year}_{month:02d}_{kernel_label}_CONUSbuf2p5deg"),
            (sigma, f"sigma_{year}_{month:02d}_{kernel_label}_CONUSbuf2p5deg"),
            (z, f"enhancementZ_{year}_{month:02d}_{kernel_label}_CONUSbuf2p5deg"),
            (mask, f"enhancement_ge2_{year}_{month:02d}_{kernel_label}_CONUSbuf2p5deg"),
        ]

        for img, out_name in exports:
            task = ee.batch.Export.image.toDrive(
                image=img,
                description=out_name,
                folder=DRIVE_EXPORT_FOLDER_GEE_PRODUCTS,
                fileNamePrefix=out_name,
                region=conus_support_geom,
                scale=EXPORT_SCALE_M,
                maxPixels=1e13,
                fileFormat="GeoTIFF",
            )
            task.start()
            tasks.append(task)
            print("Started export:", out_name)

    print(f"Submitted {len(tasks)} GEE product exports.")
    return tasks

# Example: export all kernels for the Visalia month.
# gee_tasks = start_gee_product_exports_one_month(2024, 10, KERNEL_DEG_LIST)


## 5. Local tiled processor for anomaly and enhancement

This is the main reproducible local processor.

For each monthly mean raster and kernel width, it computes:

\[
B_k(x,y) = \mathrm{median}(XCH4 \text{ in local window})
\]

\[
\Delta XCH4_k(x,y) = XCH4(x,y) - B_k(x,y)
\]

\[
Z_k(x,y) = \frac{\Delta XCH4_k(x,y)}{\max(\sigma_k(x,y), 0.1)}
\]

where \(\sigma_k\) is calculated from the local window after capping values above the local 95th percentile. The centre pixel is excluded from the local background statistics. Outputs are:

- anomaly GeoTIFF, ppb;
- sigma GeoTIFF, ppb;
- continuous enhancement Z GeoTIFF, dimensionless;
- binary mask GeoTIFF for \(Z \ge 2\).

For large kernels, use a GPU runtime and process one/few months at a time.


In [ ]:
# ============================================================
# LOCAL GPU/CPU TILED PROCESSOR
# ============================================================

if torch is None:
    raise ImportError("PyTorch is required for the local tiled processor.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

PIXEL_DEG = 0.01
TILE = 16          # reduce to 8 or 12 if GPU memory is limited
MIN_VALID_FRAC = 0.25
MIN_KEPT_FRAC_AFTER_TRIM = 0.10
NODATA_FLOAT = -9999.0
NODATA_BYTE = 0

def read_window_with_pad(src, row0, row1, col0, col1, halo):
    H, W = src.height, src.width
    rr0 = max(0, row0 - halo)
    rr1 = min(H, row1 + halo)
    cc0 = max(0, col0 - halo)
    cc1 = min(W, col1 + halo)

    arr = src.read(1, window=Window(cc0, rr0, cc1 - cc0, rr1 - rr0)).astype(np.float32)
    nod = src.nodata
    if nod is not None:
        arr[arr == nod] = np.nan

    out_h = (row1 - row0) + 2 * halo
    out_w = (col1 - col0) + 2 * halo
    out = np.full((out_h, out_w), np.nan, dtype=np.float32)

    ins_r0 = rr0 - (row0 - halo)
    ins_c0 = cc0 - (col0 - halo)
    out[ins_r0:ins_r0 + arr.shape[0], ins_c0:ins_c0 + arr.shape[1]] = arr
    return out

def write_geotiff(path, arr, src_profile, dtype="float32", nodata=NODATA_FLOAT):
    profile = src_profile.copy()
    profile.update(
        driver="GTiff",
        dtype=dtype,
        count=1,
        compress="deflate",
        tiled=True,
        nodata=nodata,
        BIGTIFF="IF_SAFER",
    )
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(arr.astype(dtype), 1)

def compute_tile_products(tile_with_halo, tile_h, tile_w, radius_px, min_valid_frac, min_kept_frac, sigma_min):
    win = 2 * radius_px + 1
    center_idx = (win * win) // 2

    x = torch.as_tensor(tile_with_halo, dtype=torch.float32, device=DEVICE)[None, None, :, :]
    patches = F.unfold(x, kernel_size=(win, win), stride=1)
    patches = patches.squeeze(0).transpose(0, 1)  # n_pixels x win*win

    center = patches[:, center_idx].clone()
    patches[:, center_idx] = float("nan")

    finite = torch.isfinite(patches)
    valid_count = finite.sum(dim=1)
    min_valid_count = int(math.ceil(min_valid_frac * (win * win - 1)))
    enough_valid = valid_count >= min_valid_count

    # local median background
    bg = torch.nanmedian(patches, dim=1).values

    # local p95; torch.nanquantile is available in recent PyTorch versions
    p95 = torch.nanquantile(patches, 0.95, dim=1)

    capped = torch.where(patches > p95[:, None], float("nan"), patches)
    kept_count = torch.isfinite(capped).sum(dim=1)
    min_kept_count = int(math.ceil(min_kept_frac * (win * win - 1)))
    enough_kept = kept_count >= min_kept_count

    sigma = torch.nanstd(capped, dim=1, unbiased=False)
    sigma_safe = torch.clamp(sigma, min=sigma_min)

    anomaly = center - bg
    z = anomaly / sigma_safe

    valid_output = torch.isfinite(center) & torch.isfinite(bg) & torch.isfinite(sigma) & enough_valid & enough_kept

    anomaly = torch.where(valid_output, anomaly, torch.full_like(anomaly, float("nan")))
    sigma = torch.where(valid_output, sigma, torch.full_like(sigma, float("nan")))
    z = torch.where(valid_output, z, torch.full_like(z, float("nan")))

    anomaly_np = anomaly.reshape(tile_h, tile_w).detach().cpu().numpy().astype(np.float32)
    sigma_np = sigma.reshape(tile_h, tile_w).detach().cpu().numpy().astype(np.float32)
    z_np = z.reshape(tile_h, tile_w).detach().cpu().numpy().astype(np.float32)

    del x, patches, center, finite, valid_count, bg, p95, capped, kept_count, sigma, sigma_safe, anomaly, z
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return anomaly_np, sigma_np, z_np

def process_one_month_kernel(mean_path, out_dir, year, month, kernel_deg, skip_existing=True):
    kernel_label = f"{int(kernel_deg)}deg"
    radius_px = int(round((kernel_deg / 2.0) / PIXEL_DEG))
    win = 2 * radius_px + 1

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    anomaly_path = out_dir / ANOM_TEMPLATE.format(year=year, month=month, kernel=kernel_label)
    sigma_path = out_dir / SIGMA_TEMPLATE.format(year=year, month=month, kernel=kernel_label)
    z_path = out_dir / Z_TEMPLATE.format(year=year, month=month, kernel=kernel_label)
    mask_path = out_dir / MASK_TEMPLATE.format(year=year, month=month, kernel=kernel_label)

    if skip_existing and all(p.exists() for p in [anomaly_path, sigma_path, z_path, mask_path]):
        print(f"Skipping existing outputs for {year}-{month:02d}, {kernel_label}")
        return anomaly_path, sigma_path, z_path, mask_path

    print(f"Processing {year}-{month:02d}, kernel={kernel_label}, radius_px={radius_px}, window={win}x{win}")
    print("Input:", mean_path)

    with rasterio.open(mean_path) as src:
        H, W = src.height, src.width
        profile = src.profile.copy()

        anomaly_full = np.full((H, W), np.nan, dtype=np.float32)
        sigma_full = np.full((H, W), np.nan, dtype=np.float32)
        z_full = np.full((H, W), np.nan, dtype=np.float32)

        for row0 in range(0, H, TILE):
            row1 = min(row0 + TILE, H)
            for col0 in range(0, W, TILE):
                col1 = min(col0 + TILE, W)

                tile_h = row1 - row0
                tile_w = col1 - col0

                tile_pad = read_window_with_pad(src, row0, row1, col0, col1, halo=radius_px)

                try:
                    anom_tile, sigma_tile, z_tile = compute_tile_products(
                        tile_pad, tile_h, tile_w, radius_px,
                        min_valid_frac=MIN_VALID_FRAC,
                        min_kept_frac=MIN_KEPT_FRAC_AFTER_TRIM,
                        sigma_min=SIGMA_MIN,
                    )
                except RuntimeError as exc:
                    if "out of memory" in str(exc).lower() and DEVICE == "cuda":
                        torch.cuda.empty_cache()
                        raise RuntimeError(
                            "GPU out of memory. Reduce TILE or process a smaller kernel/month."
                        ) from exc
                    raise

                anomaly_full[row0:row1, col0:col1] = anom_tile
                sigma_full[row0:row1, col0:col1] = sigma_tile
                z_full[row0:row1, col0:col1] = z_tile

            print(f"  completed rows {row1}/{H}")

        # write float outputs
        anomaly_write = np.where(np.isfinite(anomaly_full), anomaly_full, NODATA_FLOAT)
        sigma_write = np.where(np.isfinite(sigma_full), sigma_full, NODATA_FLOAT)
        z_write = np.where(np.isfinite(z_full), z_full, NODATA_FLOAT)

        write_geotiff(anomaly_path, anomaly_write, profile, dtype="float32", nodata=NODATA_FLOAT)
        write_geotiff(sigma_path, sigma_write, profile, dtype="float32", nodata=NODATA_FLOAT)
        write_geotiff(z_path, z_write, profile, dtype="float32", nodata=NODATA_FLOAT)

        # binary mask: 1 = Z >= threshold, 0 = NoData or not enhanced
        mask = np.zeros((H, W), dtype=np.uint8)
        mask[np.isfinite(z_full) & (z_full >= ENH_THRESHOLD)] = 1
        write_geotiff(mask_path, mask, profile, dtype="uint8", nodata=NODATA_BYTE)

    print("Saved:")
    print(" ", anomaly_path)
    print(" ", sigma_path)
    print(" ", z_path)
    print(" ", mask_path)

    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return anomaly_path, sigma_path, z_path, mask_path

def find_mean_raster(year, month):
    path = XCH4_MEAN_DIR / MEAN_TEMPLATE.format(year=year, month=month)
    if not path.exists():
        raise FileNotFoundError(f"Missing monthly mean raster: {path}")
    return path


## 6. Run local processing

Start with one month and one kernel to test paths and memory. Then process the full set.

For the thesis Visalia comparison, run October 2024 for kernels 1°–6°.


In [ ]:
# ============================================================
# TEST / CASE RUN
# ============================================================

# Example: run all Stage 3 kernels for the Visalia month.
# Large kernels can be slow; use a GPU runtime.
RUN_CASE = False

if RUN_CASE:
    for kernel_deg in KERNEL_DEG_LIST:
        mean_path = find_mean_raster(CASE_YEAR, CASE_MONTH)
        process_one_month_kernel(
            mean_path=mean_path,
            out_dir=LOCAL_PRODUCTS_DIR,
            year=CASE_YEAR,
            month=CASE_MONTH,
            kernel_deg=kernel_deg,
            skip_existing=True,
        )


In [ ]:
# ============================================================
# FULL ARCHIVE RUN: 2023-2025, ALL MONTHS, ALL KERNELS
# ============================================================
# Use with care. This is computationally expensive.
# Process in batches, for example one year or one kernel at a time.

RUN_FULL_ARCHIVE = False

if RUN_FULL_ARCHIVE:
    for year in YEARS:
        for month in MONTHS:
            mean_path = find_mean_raster(year, month)
            for kernel_deg in KERNEL_DEG_LIST:
                process_one_month_kernel(
                    mean_path=mean_path,
                    out_dir=LOCAL_PRODUCTS_DIR,
                    year=year,
                    month=month,
                    kernel_deg=kernel_deg,
                    skip_existing=True,
                )


## 7. Load Carbon Mapper plume-event records for point-to-mask comparison

This section samples the same case plume-event coordinates against each kernel mask. It is written robustly for common Carbon Mapper column names.

Required information:

- longitude;
- latitude;
- timestamp or pre-filtered case CSV.

For the Visalia example, the notebook filters the plume table to October 2024 and the bounding box defined in `CASE_BBOX`.


In [ ]:
# ============================================================
# LOAD AND FILTER CASE PLUMES
# ============================================================

def find_first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of these columns found: {candidates}")

def load_case_plumes(plumes_csv, year, month, bbox):
    df = pd.read_csv(plumes_csv)

    lon_col = find_first_existing_column(df, [
        "plume_longitude", "plume_lon", "longitude", "lon",
        "source_longitude", "point_source_longitude"
    ])
    lat_col = find_first_existing_column(df, [
        "plume_latitude", "plume_lat", "latitude", "lat",
        "source_latitude", "point_source_latitude"
    ])

    # timestamp is optional if the CSV is already pre-filtered
    timestamp_col = None
    for c in ["scene_timestamp", "datetime", "timestamp", "observation_time", "date"]:
        if c in df.columns:
            timestamp_col = c
            break

    if "gas" in df.columns:
        df = df[df["gas"].astype(str).str.upper().eq("CH4")].copy()

    if timestamp_col is not None:
        raw = df[timestamp_col].astype(str)
        dt = pd.to_datetime(raw, errors="coerce", utc=True)
        df["_year"] = dt.dt.year
        df["_month"] = dt.dt.month
        df = df[(df["_year"] == year) & (df["_month"] == month)].copy()

    min_lon, min_lat, max_lon, max_lat = bbox
    df = df[
        (df[lon_col] >= min_lon) & (df[lon_col] <= max_lon) &
        (df[lat_col] >= min_lat) & (df[lat_col] <= max_lat)
    ].copy()

    df = df.dropna(subset=[lon_col, lat_col]).copy()
    df["_lon"] = df[lon_col].astype(float)
    df["_lat"] = df[lat_col].astype(float)

    print(f"Loaded case plume-event records: {len(df)}")
    print("Longitude column:", lon_col)
    print("Latitude column:", lat_col)
    print("Timestamp column:", timestamp_col)

    return df

# Run after setting PLUMES_CSV to an existing file.
# case_plumes = load_case_plumes(PLUMES_CSV, CASE_YEAR, CASE_MONTH, CASE_BBOX)
# case_plumes.head()


## 8. Sample enhancement masks and continuous Z rasters at plume-event coordinates

For each kernel, the notebook samples:

- binary mask: whether the plume-event point falls in a retained \(Z \ge 2\) cell;
- continuous Z value at the same point.

The output summary is the table used to describe kernel sensitivity.


In [ ]:
# ============================================================
# SAMPLE CASE POINTS AGAINST KERNEL PRODUCTS
# ============================================================

def sample_raster_at_points(raster_path, points_df, value_name):
    values = []
    with rasterio.open(raster_path) as src:
        for lon, lat in zip(points_df["_lon"], points_df["_lat"]):
            try:
                row, col = src.index(lon, lat)
                if row < 0 or row >= src.height or col < 0 or col >= src.width:
                    values.append(np.nan)
                    continue
                val = src.read(1, window=Window(col, row, 1, 1))[0, 0]
                if src.nodata is not None and val == src.nodata:
                    values.append(np.nan)
                else:
                    values.append(float(val))
            except Exception:
                values.append(np.nan)
    return pd.Series(values, name=value_name)

def summarize_kernel_overlap(points_df, year, month, kernel_list=KERNEL_DEG_LIST):
    all_rows = []
    sampled = points_df.copy()

    for kernel_deg in kernel_list:
        kernel_label = f"{int(kernel_deg)}deg"
        mask_path = LOCAL_PRODUCTS_DIR / MASK_TEMPLATE.format(year=year, month=month, kernel=kernel_label)
        z_path = LOCAL_PRODUCTS_DIR / Z_TEMPLATE.format(year=year, month=month, kernel=kernel_label)

        if not mask_path.exists():
            raise FileNotFoundError(mask_path)
        if not z_path.exists():
            raise FileNotFoundError(z_path)

        mask_vals = sample_raster_at_points(mask_path, points_df, f"mask_{kernel_label}")
        z_vals = sample_raster_at_points(z_path, points_df, f"z_{kernel_label}")

        sampled[f"mask_{kernel_label}"] = mask_vals
        sampled[f"z_{kernel_label}"] = z_vals

        intersects = mask_vals.fillna(0).astype(float) >= 1
        n_intersections = int(intersects.sum())
        n_total = len(points_df)
        share = 100.0 * n_intersections / n_total if n_total else np.nan

        z_inter = z_vals[intersects & z_vals.notna()]
        z_range = ""
        if len(z_inter) > 0:
            z_range = f"{z_inter.min():.2f}-{z_inter.max():.2f}"

        all_rows.append({
            "kernel_deg": kernel_deg,
            "intersections": n_intersections,
            "total_records": n_total,
            "share_percent": share,
            "z_min_at_intersections": float(z_inter.min()) if len(z_inter) else np.nan,
            "z_max_at_intersections": float(z_inter.max()) if len(z_inter) else np.nan,
            "z_range_at_intersections": z_range,
        })

    summary = pd.DataFrame(all_rows)
    return sampled, summary

# Example:
# sampled_case, kernel_summary = summarize_kernel_overlap(case_plumes, CASE_YEAR, CASE_MONTH)
# display(kernel_summary)
# sampled_case.to_csv(SUMMARY_DIR / f"{CASE_NAME}_sampled_kernel_values.csv", index=False)
# kernel_summary.to_csv(SUMMARY_DIR / f"{CASE_NAME}_kernel_overlap_summary.csv", index=False)


## 9. Plot kernel sensitivity maps

This figure plots binary enhancement masks for all kernels and overlays the same Carbon Mapper plume-event coordinates in each panel.


In [ ]:
# ============================================================
# PLOT KERNEL MASKS WITH CASE POINTS
# ============================================================

def read_raster_for_plot(path, bbox=None):
    with rasterio.open(path) as src:
        if bbox is None:
            arr = src.read(1)
            extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
            return arr, extent

        min_lon, min_lat, max_lon, max_lat = bbox
        row_min, col_min = src.index(min_lon, max_lat)
        row_max, col_max = src.index(max_lon, min_lat)

        row0, row1 = sorted([row_min, row_max])
        col0, col1 = sorted([col_min, col_max])

        row0 = max(0, row0)
        col0 = max(0, col0)
        row1 = min(src.height - 1, row1)
        col1 = min(src.width - 1, col1)

        win = Window(col0, row0, col1 - col0 + 1, row1 - row0 + 1)
        arr = src.read(1, window=win)
        bounds = rasterio.windows.bounds(win, src.transform)
        extent = [bounds[0], bounds[2], bounds[1], bounds[3]]
        return arr, extent

def add_simple_north_arrow(ax, x=0.94, y=0.92):
    ax.annotate(
        "N", xy=(x, y), xytext=(x, y - 0.12),
        xycoords="axes fraction", textcoords="axes fraction",
        ha="center", va="center",
        arrowprops=dict(arrowstyle="-|>", lw=1.2),
        fontsize=10,
    )

def plot_case_kernel_masks(points_df, year, month, bbox, out_path=None):
    fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
    axes = axes.ravel()

    for ax, kernel_deg in zip(axes, KERNEL_DEG_LIST):
        kernel_label = f"{int(kernel_deg)}deg"
        mask_path = LOCAL_PRODUCTS_DIR / MASK_TEMPLATE.format(year=year, month=month, kernel=kernel_label)

        arr, extent = read_raster_for_plot(mask_path, bbox=bbox)
        arr_plot = np.where(arr >= 1, 1, np.nan)

        ax.imshow(arr_plot, extent=extent, origin="upper", interpolation="nearest")
        ax.scatter(points_df["_lon"], points_df["_lat"], s=35, edgecolor="black", linewidth=0.5, label="Carbon Mapper plume-event record")
        ax.set_title(f"{int(kernel_deg)}° local-background window")
        ax.set_xlabel("Longitude (°)")
        ax.set_ylabel("Latitude (°)")
        ax.grid(alpha=0.2)

    add_simple_north_arrow(axes[2])

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=2)
    fig.suptitle(f"{CASE_NAME}: Sentinel-5P threshold masks (Z ≥ {ENH_THRESHOLD})", y=1.02)

    if out_path:
        fig.savefig(out_path, dpi=300, bbox_inches="tight")
        print("Saved figure:", out_path)

    plt.show()

# Example:
# plot_case_kernel_masks(case_plumes, CASE_YEAR, CASE_MONTH, CASE_BBOX, FIGURE_DIR / f"{CASE_NAME}_kernel_masks.png")


## 10. Plot intermediate products for one kernel

This figure shows the monthly mean XCH4, anomaly and standardised enhancement for one selected kernel, together with the binary \(Z \ge 2\) mask.


In [ ]:
# ============================================================
# PLOT INTERMEDIATE PRODUCTS FOR ONE KERNEL
# ============================================================

def plot_intermediate_products(points_df, year, month, kernel_deg=5, bbox=CASE_BBOX, out_path=None):
    kernel_label = f"{int(kernel_deg)}deg"
    mean_path = find_mean_raster(year, month)
    anom_path = LOCAL_PRODUCTS_DIR / ANOM_TEMPLATE.format(year=year, month=month, kernel=kernel_label)
    z_path = LOCAL_PRODUCTS_DIR / Z_TEMPLATE.format(year=year, month=month, kernel=kernel_label)
    mask_path = LOCAL_PRODUCTS_DIR / MASK_TEMPLATE.format(year=year, month=month, kernel=kernel_label)

    paths = [
        (mean_path, "Monthly mean XCH4", "ppb"),
        (anom_path, f"{int(kernel_deg)}° anomaly", "ppb"),
        (z_path, f"{int(kernel_deg)}° standardised enhancement Z", "Z"),
        (mask_path, f"Threshold mask Z ≥ {ENH_THRESHOLD}", "binary"),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
    axes = axes.ravel()

    for ax, (path, title, unit) in zip(axes, paths):
        arr, extent = read_raster_for_plot(path, bbox=bbox)
        if "mask" in title.lower():
            arr_plot = np.where(arr >= 1, 1, np.nan)
        else:
            arr = arr.astype(float)
            arr[arr == NODATA_FLOAT] = np.nan
            arr_plot = arr

        im = ax.imshow(arr_plot, extent=extent, origin="upper", interpolation="nearest")
        ax.scatter(points_df["_lon"], points_df["_lat"], s=35, edgecolor="black", linewidth=0.5)
        ax.set_title(title)
        ax.set_xlabel("Longitude (°)")
        ax.set_ylabel("Latitude (°)")
        ax.grid(alpha=0.2)
        cbar = fig.colorbar(im, ax=ax, shrink=0.78)
        cbar.set_label(unit)

    add_simple_north_arrow(axes[1])
    fig.suptitle(f"{CASE_NAME}: Stage 3 intermediate products", y=1.02)

    if out_path:
        fig.savefig(out_path, dpi=300, bbox_inches="tight")
        print("Saved figure:", out_path)

    plt.show()

# Example:
# plot_intermediate_products(case_plumes, CASE_YEAR, CASE_MONTH, kernel_deg=5, bbox=CASE_BBOX, out_path=FIGURE_DIR / f"{CASE_NAME}_intermediate_products_5deg.png")


## 11. QA checks

Required checks before reporting results:

- verify all rasters have identical CRS, transform and dimensions;
- verify NoData is distinct from non-enhanced cells;
- verify plume coordinates are inside raster bounds;
- verify the same plume-event coordinates are sampled against every kernel;
- verify the table counts match the plotted points.


In [ ]:
# ============================================================
# QA HELPERS
# ============================================================

def inspect_raster(path):
    with rasterio.open(path) as src:
        arr = src.read(1, masked=True)
        print("Path:", path)
        print("  CRS:", src.crs)
        print("  Transform:", src.transform)
        print("  Shape:", (src.height, src.width))
        print("  Dtype:", src.dtypes[0])
        print("  Nodata:", src.nodata)
        print("  Bounds:", src.bounds)
        print("  Valid cells:", int(np.ma.count(arr)))
        if np.ma.count(arr) > 0:
            print("  Min/max:", float(arr.min()), float(arr.max()))

def compare_raster_grid(paths):
    meta = []
    for p in paths:
        with rasterio.open(p) as src:
            meta.append((str(src.crs), tuple(src.transform), src.width, src.height))
    first = meta[0]
    ok = all(m == first for m in meta)
    print("All grids identical:", ok)
    if not ok:
        for p, m in zip(paths, meta):
            print(p, m)
    return ok

# Example:
# paths_to_check = [
#     find_mean_raster(CASE_YEAR, CASE_MONTH),
#     LOCAL_PRODUCTS_DIR / Z_TEMPLATE.format(year=CASE_YEAR, month=CASE_MONTH, kernel="5deg"),
#     LOCAL_PRODUCTS_DIR / MASK_TEMPLATE.format(year=CASE_YEAR, month=CASE_MONTH, kernel="5deg"),
# ]
# compare_raster_grid(paths_to_check)
# for p in paths_to_check:
#     inspect_raster(p)


## 12. Generated outputs

The notebook generates the following products:

```text
XCH4_mean_YYYY_MM_CONUSbuf2p5deg.tif
anomaly_YYYY_MM_Kdeg_CONUSbuf2p5deg.tif
sigma_YYYY_MM_Kdeg_CONUSbuf2p5deg.tif
enhancementZ_YYYY_MM_Kdeg_CONUSbuf2p5deg.tif
enhancement_ge2_YYYY_MM_Kdeg_CONUSbuf2p5deg.tif
Visalia_October_2024_sampled_kernel_values.csv
Visalia_October_2024_kernel_overlap_summary.csv
Visalia_October_2024_kernel_masks.png
Visalia_October_2024_intermediate_products_5deg.png
```

Large raster products are stored in the external thesis dataset described in
`docs/data-availability.md`.
